In [1]:
import re, torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from evalplus.data import get_human_eval_plus, get_mbpp_plus, write_jsonl

In [4]:
CKPT      = "./poc_checkpoints_3B_QAT4bit_Qwen_7B/final_quantized"   # adjust per run
TOKENIZER = "./StudentModel/Qwen2.5-Coder-3B/"
DATASET   = "humaneval"        # or "mbpp"
OUT       = f"./{DATASET}_Qwen-2.5-Coder-QAT4bit-7B-3B.jsonl"
DEVICE    = "cuda:0"

In [5]:
model = AutoModelForCausalLM.from_pretrained(
    CKPT, dtype=torch.bfloat16, trust_remote_code=True).to(DEVICE)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

In [6]:
model.eval()

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 2048)
    (layers): ModuleList(
      (0-35): 36 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=True)
          (k_proj): Linear(in_features=2048, out_features=256, bias=True)
          (v_proj): Linear(in_features=2048, out_features=256, bias=True)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=2048, out_features=11008, bias=False)
          (up_proj): Linear(in_features=2048, out_features=11008, bias=False)
          (down_proj): Linear(in_features=11008, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((2048,), eps=1e-06)
    (ro

In [7]:
tok = AutoTokenizer.from_pretrained(TOKENIZER)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

In [8]:
def clean(text):
    for pat in [r'\n\ndef ', r'\n\nclass ', r'\n\nasync def ',
                r'\nif __name__', r'\n\nprint\(', r'\n\n#']:
        m = re.search(pat, text)
        if m:
            text = text[:m.start()]
    return text

In [9]:
problems = get_human_eval_plus() if DATASET == "humaneval" else get_mbpp_plus()
print(f"{DATASET}: {len(problems)} problems")
samples = []
with torch.no_grad():
    for task_id, problem in tqdm(problems.items(), total=len(problems), desc=f"generating {DATASET}"):
        inp = tok(problem["prompt"], return_tensors="pt").to(DEVICE)
        out = model.generate(
            **inp,
            max_new_tokens=512,
            do_sample=False,
            pad_token_id=tok.pad_token_id,
            eos_token_id=tok.eos_token_id,
        )
        gen = tok.decode(out[0][inp["input_ids"].shape[1]:],
                         skip_special_tokens=True)
        samples.append({"task_id": task_id, "solution": problem["prompt"] + clean(gen)})

write_jsonl(OUT, samples)
print(f"wrote {len(samples)} completions to {OUT}")

humaneval: 164 problems


generating humaneval:   0%|          | 0/164 [00:00<?, ?it/s]

wrote 164 completions to ./humaneval_Qwen-2.5-Coder-QAT4bit-7B-3B.jsonl


In [8]:
evalplus.evaluate --dataset humaneval --samples humaneval_Qwen-2.5-Coder-7B.jsonl

STDERR:
 
0it [00:00, ?it/s]
0it [00:00, ?it/s]
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/home/vamsitej/.local/lib/python3.13/site-packages/evalplus/evaluate.py", line 363, in <module>
    main()
    ~~~~^^
  File "/home/vamsitej/.local/lib/python3.13/site-packages/evalplus/evaluate.py", line 359, in main
    Fire(evaluate)
    ~~~~^^^^^^^^^^
  File "/home/vamsitej/.local/lib/python3.13/site-packages/fire/core.py", line 135, in Fire
    component_trace = _Fire(component, args, parsed_flag_args, context, name)
  File "/home/vamsitej/.local/lib/python3.13/site-packages/fire/core.py", line 468, in _Fire
    component, remaining_args = _CallAndUpdateTrace(
                                ~~~~~~~~~~~~~~~~~~~^
        component,
        ^^^^^^^^^^
    ...<2 lines>...
        treatment='class' if is_class else 'routine',
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        targe